# NB 59 — Subspace vs frame steering (56/57 vs the FRH pipeline)

Plots the subspace experiments against the **equivalent frame-based runs** on the
same pool (NB 32), the same 10 pairs (NB 33), the same recipe (`k=4, steps=24`,
bf16 Llama-3.1-8B), and the same fluency dial `λ ∈ {0, 0.4}`:

| representation | singles | blend (n=2) |
|---|---|---|
| **frame** (FRH pipeline) | NB 44 | NB 33 (λ=0) + NB 40 (λ=0.4) |
| **subspace** pooled / hierarchical | NB 57 (multilingual) | NB 57 |

The frame runs use the 8-language set at `MIN_LEMMAS=11`, so they line up with the
**multilingual** subspace run (NB 57); the English subspace run (NB 56) is listed
in the table for reference.

**Metric note.** Frame expression Δ (sliding-window frame correlation) and
subspace Δ (`‖Bᵀh‖`) are different units, so the figure leads with the
**scale-invariant** metrics — positive-Δ rate (singles), both-improved rate
(blend), and perplexity ratio — which are directly comparable across
representations. Raw Δ is shown separately, with that caveat. GPU-free.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

R = Path("resources")
LAMBDAS = [0.0, 0.4]


def load(path):
    return [json.loads(ln) for ln in (R / path).read_text().strip().split("\n") if ln]


F44 = load("44_singles_lambda_bf16.jsonl")   # frame singles (λ sweep)
F33 = load("33_scaling3_bf16.jsonl")         # frame blend, λ=0 (logged lam=None)
F40 = load("40_greedy_lambda.jsonl")         # frame blend, λ=0.4
S57 = load("57_subspace_ml.jsonl")           # subspace, multilingual
S56 = load("56_subspace_pooled_vs_hier.jsonl")  # subspace, English

In [ ]:
def singles_metrics(recs):
    d = np.array([r["expression"][r["config"]["concept"]]["delta"] for r in recs])
    ppl = np.array([r["ppl_ratio"] for r in recs])
    return {"delta": d.mean(), "positive": (d > 0).mean(), "ppl": ppl.mean(), "n": len(d)}


def blend_metrics(recs):
    joint, both, ppl = [], [], []
    for r in recs:
        c = r["config"]
        ds = [r["expression"][c["a"]]["delta"], r["expression"][c["b"]]["delta"]]
        joint.append(min(ds))
        both.append(all(x > 0 for x in ds))
        ppl.append(r["ppl_ratio"])
    return {
        "joint": float(np.mean(joint)),
        "both_improved": float(np.mean(both)),
        "ppl": float(np.mean(ppl)),
        "n": len(joint),
    }


def frame_singles(lam):
    return [r for r in F44 if r["config"]["method"] == "single" and r["config"]["lam"] == lam]


def frame_blend(lam):
    if lam == 0.0:  # NB 33 blend n=2 (logged lam=None)
        return [r for r in F33 if r["config"].get("method") == "blend" and r["config"].get("n") == 2]
    return [r for r in F40 if r["config"].get("method") == "blend" and r["config"].get("n") == 2 and r["config"].get("lam") == lam]


def sub(recs, arm, mode, lam):
    return [
        r for r in recs
        if r["config"]["arm"] == arm and r["config"]["mode"] == mode and r["config"]["lam"] == lam
    ]


# Series -> (singles selector, blend selector). Frame first (the baseline).
SERIES = {
    "frame": (lambda lam: frame_singles(lam), lambda lam: frame_blend(lam)),
    "subspace-pooled": (
        lambda lam: sub(S57, "pooled", "single", lam),
        lambda lam: sub(S57, "pooled", "blend", lam),
    ),
    "subspace-hier": (
        lambda lam: sub(S57, "hierarchical", "single", lam),
        lambda lam: sub(S57, "hierarchical", "blend", lam),
    ),
    "subspace-pooled (En)": (
        lambda lam: sub(S56, "pooled", "single", lam),
        lambda lam: sub(S56, "pooled", "blend", lam),
    ),
    "subspace-hier (En)": (
        lambda lam: sub(S56, "hierarchical", "single", lam),
        lambda lam: sub(S56, "hierarchical", "blend", lam),
    ),
}

rows = []
for name, (sing, bl) in SERIES.items():
    for lam in LAMBDAS:
        s, b = singles_metrics(sing(lam)), blend_metrics(bl(lam))
        rows.append({
            "series": name, "lam": lam,
            "sing_delta": s["delta"], "sing_positive": s["positive"], "sing_ppl": s["ppl"],
            "blend_joint": b["joint"], "blend_both": b["both_improved"], "blend_ppl": b["ppl"],
            "sing_n": s["n"], "blend_n": b["n"],
        })
tbl = pd.DataFrame(rows)
tbl.to_csv(R / "59_frame_vs_subspace.csv", index=False)
pd.set_option("display.width", 200)
print(tbl.round(3).to_string(index=False))

## Figure — scale-invariant comparison

Multilingual series only (matched to the frame baseline): frame vs subspace-pooled vs subspace-hierarchical.

In [ ]:
PLOT = ["frame", "subspace-pooled", "subspace-hier"]
colors = {"frame": "tab:red", "subspace-pooled": "tab:gray", "subspace-hier": "tab:blue"}
panels = [
    ("sing_positive", "singles: positive-Δ rate"),
    ("blend_both", "blend: both-improved rate"),
    ("blend_ppl", "blend: perplexity ratio"),
]
x = np.arange(len(PLOT))
fig, axes = plt.subplots(len(LAMBDAS), len(panels), figsize=(4 * len(panels), 7))
for row, lam in enumerate(LAMBDAS):
    for col, (metric, label) in enumerate(panels):
        ax = axes[row, col]
        vals = [tbl[(tbl.series == s) & (tbl.lam == lam)][metric].iloc[0] for s in PLOT]
        ax.bar(x, vals, color=[colors[s] for s in PLOT])
        ax.set_xticks(x)
        ax.set_xticklabels(["frame", "pooled", "hier"])
        ax.set_title(f"{label} (λ={lam})")
        ax.axhline(0, color="gray", lw=0.6)
        if metric == "blend_ppl":
            ax.axhline(2.5, color="red", ls=":", lw=0.8)
plt.suptitle("Subspace vs frame steering (multilingual, matched recipe)")
plt.tight_layout()
plt.savefig(R / "59_frame_vs_subspace.png", dpi=150)
plt.show()

## Raw Δ (secondary — different units across representations)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (metric, label) in zip(axes, [("sing_delta", "singles Δ"), ("blend_joint", "blend joint Δ")]):
    width = 0.38
    for j, lam in enumerate(LAMBDAS):
        vals = [tbl[(tbl.series == s) & (tbl.lam == lam)][metric].iloc[0] for s in PLOT]
        ax.bar(x + (j - 0.5) * width, vals, width, label=f"λ={lam}",
               color=["lightsteelblue", "tab:blue"][j])
    ax.set_xticks(x)
    ax.set_xticklabels(["frame", "pooled", "hier"])
    ax.set_title(label)
    ax.legend()
plt.suptitle("Raw Δ — magnitudes NOT directly comparable (frame corr. vs ‖Bᵀh‖)")
plt.tight_layout()
plt.savefig(R / "59_frame_vs_subspace_rawdelta.png", dpi=150)
plt.show()

## Findings

- **Singles.** Read as positive-Δ rate (scale-invariant): frame and both
  subspace builds all steer nearly every concept in the intended direction; the
  representations are close, and the λ dial trims positive-rate similarly.
- **Blend.** Both-improved rate is the fair cross-representation composition
  metric. Compare frame's Family-1 blend (`Concept.average`) against the subspace
  `ConceptSubspace.blend` at n=2.
- **Fluency.** Perplexity ratio is in the same units for all three; it shows
  whether subspace steering pays a different fluency cost than the frame pipeline
  at matched λ.
- Raw Δ is plotted separately only to show within-representation trends; frame
  correlation and `‖Bᵀh‖` are different units, so cross-representation Δ
  magnitudes should not be read as "stronger/weaker".